# Dataset Prep

### Import Libraries

In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

### Load Dataset

In [7]:
hurricanes = pd.read_csv("data/cleaned_data.csv") # renamed kaggle csv from storms to hurricane_data
print(hurricanes.columns.tolist())

['name', 'year', 'month', 'day', 'hour', 'lat', 'long', 'status', 'category', 'wind', 'pressure', 'tropicalstorm_force_diameter', 'hurricane_force_diameter', 'hurricane_class']


### Create Modeling Dataset

In [18]:
predictors = [
    "year",
    "month",
    "day",
    "hour",
    "lat",
    "long",
    "wind",
    "pressure",
    "tropicalstorm_force_diameter"
]

target = "hurricane_class"

model_data = hurricanes[predictors + [target]].dropna()
X = model_data[predictors]
y = model_data[target]
X = sm.add_constant(X)

print(f"Observations used: {len(model_data)}")
print(X.shape)
print(y.shape)

Observations used: 5100
(5100, 10)
(5100,)


### Logistic Regression Helper Function

In [22]:
def fit_logistic_model(X, y):
    model = sm.Logit(y, X)
    result = model.fit(method="bfgs", disp=False, maxiter=100)
    return result

# Feature Selection

### Forward Selection (AIC)

In [ ]:
def forward_selection_aic(X, y):

    remaining = list(X.columns)
    remaining.remove("const")

    selected = ["const"]
    current_aic = np.inf

    while remaining:

        scores = []

        for variable in remaining:
            candidate_vars = selected + [variable]
            result = fit_logistic_model(
                X[candidate_vars],
                y
            )

            scores.append(
                (result.aic, variable)
            )

        best_aic, best_variable = min(scores)

        if best_aic < current_aic:
            selected.append(best_variable)
            remaining.remove(best_variable)
            current_aic = best_aic

        else:
            break

    final_model = fit_logistic_model(
        X[selected],
        y
    )

    return final_model, selected

In [24]:
forward_aic_model, forward_aic_vars = forward_selection_aic(X, y)

print("Forward AIC Selected Variables:")
print(forward_aic_vars)

print("\nAIC:", forward_aic_model.aic)
print("BIC:", forward_aic_model.bic)

Forward AIC Selected Variables:
['const', 'wind']

AIC: 4.00055787517988
BIC: 17.074549512604715


/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/

### Backward Selection (AIC)

In [ ]:
def backward_selection_aic(X, y):

    selected = list(X.columns)

    current_model = fit_logistic_model(
        X[selected],
        y
    )

    current_aic = current_model.aic

    while len(selected) > 1:

        scores = []

        for variable in selected:
            if variable == "const":
                continue

            candidate_vars = selected.copy()
            candidate_vars.remove(variable)

            result = fit_logistic_model(
                X[candidate_vars],
                y
            )

            scores.append(
                (result.aic, variable)
            )

        best_aic, variable_to_remove = min(scores)

        if best_aic < current_aic:
            selected.remove(variable_to_remove)
            current_aic = best_aic

        else:
            break

    final_model = fit_logistic_model(
        X[selected],
        y
    )

    return final_model, selected

In [26]:
backward_aic_model, backward_aic_vars = backward_selection_aic(X, y)

print("Backward AIC Selected Variables:")
print(backward_aic_vars)

print("\nAIC:", backward_aic_model.aic)
print("BIC:", backward_aic_model.bic)

/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/

Backward AIC Selected Variables:
['const', 'wind']

AIC: 4.00055787517988
BIC: 17.074549512604715


/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/

### Stepwise Selection (BIC)

In [ ]:
def stepwise_selection_bic(X, y):

    remaining = list(X.columns)
    remaining.remove("const")

    selected = ["const"]

    current_bic = np.inf
    changed = True

    while changed:
        changed = False

        # Forward Step
        forward_scores = []

        for variable in remaining:
            candidate_vars = selected + [variable]
            result = fit_logistic_model(
                X[candidate_vars],
                y
            )

            forward_scores.append(
                (result.bic, variable)
            )

        if forward_scores:
            best_bic, best_variable = min(forward_scores)
            if best_bic < current_bic:
                selected.append(best_variable)
                remaining.remove(best_variable)

                current_bic = best_bic
                changed = True

        # Backward Step
        backward_scores = []

        for variable in selected:
            if variable == "const":
                continue

            candidate_vars = selected.copy()
            candidate_vars.remove(variable)

            result = fit_logistic_model(
                X[candidate_vars],
                y
            )

            backward_scores.append(
                (result.bic, variable)
            )

        if backward_scores:
            best_bic, variable_to_remove = min(backward_scores)
            if best_bic < current_bic:
                selected.remove(variable_to_remove)
                remaining.append(variable_to_remove)
                current_bic = best_bic
                changed = True

    final_model = fit_logistic_model(
        X[selected],
        y
    )

    return final_model, selected

In [28]:
bic_model, bic_vars = stepwise_selection_bic(X, y)

print("Stepwise BIC Selected Variables:")
print(bic_vars)

print("\nAIC:", bic_model.aic)
print("BIC:", bic_model.bic)

Stepwise BIC Selected Variables:
['const', 'wind']

AIC: 4.00055787517988
BIC: 17.074549512604715


/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/jessicawentworth/Desktop/STA6704/Group Project/hurricane-analysis/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/Users/jessicawentworth/Desktop/STA6704/Group Project/

### Final Comparison Table

In [29]:
selection_results = pd.DataFrame({
    "Method": [
        "Forward AIC",
        "Backward AIC",
        "Stepwise BIC"
    ],
    "Selected Variables": [
        ", ".join(forward_aic_vars),
        ", ".join(backward_aic_vars),
        ", ".join(bic_vars)
    ],
    "AIC": [
        round(forward_aic_model.aic, 2),
        round(backward_aic_model.aic, 2),
        round(bic_model.aic, 2)
    ],
    "BIC": [
        round(forward_aic_model.bic, 2),
        round(backward_aic_model.bic, 2),
        round(bic_model.bic, 2)
    ]
})

selection_results

,Method,Selected Variables,AIC,BIC
0,Forward AIC,"const, wind",4.0,17.07
1,Backward AIC,"const, wind",4.0,17.07
2,Stepwise BIC,"const, wind",4.0,17.07


NOTES: Traditional feature selection was performed using Forward AIC, Backward AIC, and Stepwise BIC logistic regression. All three methods selected wind speed as the sole predictor. This result is consistent with hurricane classification criteria, as hurricane intensity categories are primarily determined by sustained wind speed. Additional variables, including pressure, location, temporal factors, and storm size, did not provide sufficient incremental predictive value once wind speed was included in the model